In [10]:
import typing
from abc import abstractmethod
from paths import UrlCompatible,asUrl

class DictDataMapping:
    def __init__(self,
        memberName:str,
        encodedName:typing.Optional[str]=None,
        decoder:typing.Optional[typing.Callable]=None,
        encoder:typing.Optional[typing.Callable]=None,
        default:typing.Optional[typing.Any]=None):
        """ """
        self.memberName=memberName
        self.encodedName=encodedName
        self.decoder=decoder
        self.encoder=encoder
        self.default=default

    def __repr__(self):
        if self.encodedName is not None:
            return f'{self.memberName} -> {self.encodedName}'
        return f'{self.memberName}'

class DictIoBase:
    EXTENSIONS=[]
    MIME_TYPES=[]

    @abstractmethod
    def save(self,data:typing.Dict[str,typing.Any],location:UrlCompatible)->None:
        pass

    def canSave(self,extension:str,mime:str)->bool:
        if extension in self.EXTENSIONS:
            return True
        if mime in self.MIME_TYPES:
            return True
        return False

    @abstractmethod
    def load(self,location:UrlCompatible)->typing.Dict[str,typing.Any]:
        pass

    def canLoad(self,extension:str,mime:str)->bool:
        if extension in self.EXTENSIONS:
            return True
        if mime in self.MIME_TYPES:
            return True
        return False


class JsonDictIo(DictIoBase):
    EXTENSIONS=['.json']
    MIME_TYPES=['text/json']
    def save(self,data:typing.Dict[str,typing.Any],location:UrlCompatible)->None:
        import json
        s=json.dumps(data)
        asUrl(location).write(s)
    def load(self,location:UrlCompatible)->typing.Dict[str,typing.Any]:
        import json
        data=asUrl(location).readString()
        return json.loads(data)

class XmlDictIo(DictIoBase):
    EXTENSIONS=['.xml']
    MIME_TYPES=['text/xml']
    def save(self,data:typing.Dict[str,typing.Any],location:UrlCompatible):
        pass
    def load(self,location:UrlCompatible)->typing.Dict[str,typing.Any]:
        return {}

MappingsCompatible=typing.Union[
    str,
    typing.Dict[str,str],
    DictDataMapping,
    typing.Iterable["MappingsCompatible"]]

class DictIoData:
    _DICT_DATA_IO:typing.List[DictIoBase]=[]

    def __init__(self,*args,**kwargs):
        if not self._DICT_DATA_IO:
            for saverName in ('JsonDictIo','YamlDictIo','XmlDictIo'):
                saver=globals().get(saverName)
                if saver is not None:
                    self._DICT_DATA_IO.append(saver())
        self._mappings:typing.List[DictDataMapping]=[]
        if args:
            self._addMappings(args) # type: ignore
        if kwargs:
            self._addMappings(kwargs) # type: ignore
        if not self._mappings:
            vals=[]
            def gettersForClass(clazz:typing.Type):
                for k,v in clazz.__dict__.items():
                    if k.startswith('_'):
                        continue
                    if hasattr(v,'__call__'):
                        continue
                    vals.append(k)
                for parent in clazz.__bases__:
                    if parent is object or parent is DictIoData:
                        continue
                    gettersForClass(parent)
            gettersForClass(self.__class__)
            for k,v in self.__dict__.items():
                if k.startswith('_'):
                    continue
                if hasattr(v,'__call__'):
                    continue
                vals.append(k)
            self._addMappings(vals)

    def _encodeMappings(self)->typing.Dict[str,typing.Any]:
        """
        get the value mapped to a k:v dict
        """
        ret:typing.Dict[str,typing.Any]={}
        for mapping in self._mappings:
            if mapping.encodedName is not None:
                k=mapping.encodedName
            else:
                k=mapping.memberName
            v=getattr(self,mapping.memberName)
            if hasattr(v,'__call__'):
                v=v()
            if mapping.encoder is not None:
                v=mapping.encoder(v)
            elif not isinstance(v,(str,int,float)):
                v=str(v)
            ret[k]=v
        return ret

    def _decodeMappings(self,data:typing.Dict[str,typing.Any]):
        """
        set the value mapped from a k:v dict
        """
        for mapping in self._mappings:
            if mapping.encodedName is not None:
                k=mapping.encodedName
            else:
                k=mapping.memberName
            v=data.get(k,mapping.default)
            if mapping.decoder is not None:
                v=mapping.decoder(v)
            member=getattr(self,mapping.memberName)
            if member is not None and hasattr(member,'__call__'):
                member(v)
            else:
                setattr(self,mapping.memberName,v)

    def _addMappings(self,mappings:MappingsCompatible):
        """
        Add more mappings
        """
        if isinstance(mappings,DictDataMapping):
            self._mappings.append(mappings)
        elif isinstance(mappings,str):
            self._mappings.append(DictDataMapping(mappings))
        elif isinstance(mappings,dict):
            for k,v in mappings.items():
                self._mappings.append(DictDataMapping(str(k),str(v)))
        elif hasattr(mappings,'__iter__'):
            for mapping in mappings:
                self._addMappings(mapping)
        else:
            self._mappings.append(DictDataMapping(str(mappings)))

    @property
    def EXTENSIONS(self)->typing.List[str]:
        ret=[]
        for saver in self._DICT_DATA_IO:
            ret.extend(saver.EXTENSIONS)
        return ret
    @property
    def MIME_TYPES(self)->typing.List[str]:
        ret=[]
        for saver in self._DICT_DATA_IO:
            ret.extend(saver.MIME_TYPES)
        return ret

    def canSave(self,extension:str,mime:str):
        for io in self._DICT_DATA_IO:
            if io.canSave(extension,mime):
                return True
        return False

    def canLoad(self,extension:str,mime:str):
        for io in self._DICT_DATA_IO:
            if io.canLoad(extension,mime):
                return True
        return False

    def saveDictData(self,location:UrlCompatible,mimeType:typing.Optional[str]=None):
        location=asUrl(location)
        for io in self._DICT_DATA_IO:
            if mimeType is not None:
                if not io.canSave('',mimeType):
                    continue
            elif location.extension:
                if not io.canSave(f'.{location.extension}',''):
                    continue
            io.save(self._encodeMappings(),location)
            return
        raise FileNotFoundError('Cannot save this type of data')
    save=saveDictData

    def loadDictData(self,location:UrlCompatible,mimeType:typing.Optional[str]=None):
        location=asUrl(location)
        for io in self._DICT_DATA_IO:
            if mimeType is not None:
                if not io.canLoad('',mimeType):
                    continue
            elif location.extension:
                if not io.canLoad(f'.{location.extension}',''):
                    continue
            return io.load(location)
        raise FileNotFoundError('Cannot save this type of data')
    load=loadDictData


class Parent(DictIoData):
    
    d=5

    def __init__(self):
        DictIoData.__init__(self)

class Derived(Parent):
    def __init__(self):
        self.a:str='b'
        self.b:str='b'
        Parent.__init__(self)

    @property
    def c(self):
        return 'c'

s=Derived()
s.save('file://./foo.json')

TODO: setting of fragment is experimental
TODO: setting of fragment is experimental


In [63]:
import pathlib
from abc import ABC

class A:

    def __new__(cls,manifestation:str,*args,**kwargs):
        if manifestation=='A':
            return super(A,cls).__new__(A)
        return super(A,cls).__new__(B)

    def __init__(self,manifestation:str,*args,**kwargs):
        self.manifestation=manifestation

    @property
    def whoami(self):
        return 'A'

class B(A
    ,ABC.register(pathlib.Path)
    ):
    
    def __init__(self,manifestation:str,cSpecific="hi"):
        A.__init__(self,manifestation)
        self.cSpecific=cSpecific

    @property
    def whoami(self):
        return 'B'

a=A("A","yo")
b=A("B","yo")
b2=B("B","yo")
a.whoami,a.manifestation,'|',b.whoami,b.manifestation,'|',b2.whoami,b2.manifestation

('A', 'A', '|', 'B', 'B', '|', 'B', 'B')

In [45]:
from abc import ABC
import pathlib

class ActLikeString(ABC.register(str)):
    @property
    def _actualString(self)->str:
        """
        the actual string we are operating upon
        """
        return ''
    @_actualString.setter
    def _actualString(self,s:str):
        _=s
    #def __repr__(self):
    #    return self._actualString
    def split(self,sep=None)->typing.List[typing.Self]:
        return [self.__class__(s) for s in super().split(sep)]

class ActLikeInt(ABC.register(int)):
    @property
    def _actualInt(self)->int:
        """
        the actual string we are operating upon
        """
        return 0
    @_actualInt.setter
    def _actualInt(self,n:str):
        _=n


class Name(ActLikeString):
    def __init__(self,s):
        self._s='splat'
    @property
    def _actualString(self)->str:
        """
        the actual string we are operating upon
        """
        return self._s
    @_actualString.setter
    def _actualString(self,s:str):
        self._s=s


class Value(ActLikeString):
    def __init__(self,s):
        if isinstance(s,str):
            n=int(s)
        elif isinstance(s,int):
            n=s
            s=str(s)
        else:
            n=int(s)
            s=str(s)
        self._s=s
        self._n=n
    @property
    def _actualString(self)->str:
        """
        the actual string we are operating upon
        """
        return self._s
    @_actualString.setter
    def _actualString(self,s:str):
        self._s=s
        self._n=int(s)
    @property
    def _actualInt(self)->int:
        """
        the actual string we are operating upon
        """
        return self._n
    @_actualInt.setter
    def _actualInt(self,n:str):
        self._n=n

    
n=Name("Bill The Cat")
parts=n.split()
type(n),parts,type(parts[0])


(__main__.Name, ['Bill', 'The', 'Cat'], __main__.Name)

In [ ]:
#!/usr/bin/env python3
"""
c_ast_to_json.py

Parse a C file with pycparser and dump its AST to JSON, including
start/end line+column ranges.

Usage:
    python c_ast_to_json.py input.c output.json
"""
import typing
import os
import pathlib
import glob
import json
import subprocess
from pycparser import parse_file, c_parser, c_ast


# TODO: once we are outside of jupyter-land, need to base this upon __file__
MSVC_STUBS_FILE=pathlib.Path(r'D:\python\jsonSerializeable\chunder.ipynb').parent/'msvc_stubs.h'


def get_coord_tuple(coord):
    if coord is None:
        return None
    return (coord.line, coord.column)


def compute_span(node):
    """
    Compute (min_line, min_col), (max_line, max_col) across this node's subtree.
    """
    min_line = min_col = float("inf")
    max_line = max_col = -1

    stack = [node]
    while stack:
        cur = stack.pop()
        if hasattr(cur, "coord") and cur.coord:
            if isinstance(cur.coord.line, int):
                ln, col = cur.coord.line, cur.coord.column
                if ln < min_line or (ln == min_line and col < min_col):
                    min_line, min_col = ln, col
                if ln > max_line or (ln == max_line and col > max_col):
                    max_line, max_col = ln, col
        for _, child in cur.children():
            stack.append(child)

    if max_line == -1:
        return None, None
    return (min_line, min_col), (max_line, max_col)


def ast_to_dict(node:c_ast.FileAST):
    if node is None:
        return None

    # node type
    d = {"type": node.__class__.__name__}

    # attributes
    attrs = {}
    for name, val in vars(node).items():
        if name in ("coord",):  # skip coord, handled separately
            continue
        if isinstance(val, (str, int, float)):
            attrs[name] = val
        elif isinstance(val, list) and all(isinstance(x, (str, int, float)) for x in val):
            attrs[name] = val
    if attrs:
        d["attrs"] = attrs
    # location span
    start, end = compute_span(node)
    if start:
        d["start"] = {"line": start[0], "col": start[1]}
    if end:
        d["end"] = {"line": end[0], "col": end[1]}
    # children
    children = []
    for name, child in node.children():
        children.append({"name": name, "node": ast_to_dict(child)})
    if children:
        d["children"] = children
    return d


def ast_to_json_string(ast:c_ast.FileAST)->str:
    return json.dumps(ast_to_dict(ast))


def c_file_to_ast(
    filename:pathlib.Path,
    includeDirs:typing.Iterable[pathlib.Path],
    defines:typing.Optional[typing.Dict[str,typing.Optional[str]]]=None,
    preprocessExternally:bool=True):
    """ """
    if preprocessExternally:
        filename=preprocessCFile(filename,includeDirs,defines,force=True)
    try:
        ast = parse_file(filename, use_cpp=True)
    except Exception:
        parser = c_parser.CParser()
        with open(filename, "r", encoding="utf-8") as f:
            code = f.read()
        ast = parser.parse(code, filename=filename)
    return ast

def c_file_to_json(
    filename:pathlib.Path,
    includeDirs:typing.Iterable[pathlib.Path],
    defines:typing.Optional[typing.Dict[str,typing.Optional[str]]]=None,
    preprocessExternally:bool=True
    )->typing.Dict[str,typing.Any]:
    """
    Parse a c file and store into a json-compatible object
    """
    ast=c_file_to_ast(filename,
        includeDirs,
        defines,
        preprocessExternally)
    return {'parse_tree':ast.ext}
    tree = ast_to_dict(ast)
    return tree

def c_file_to_json_string(
    filename:pathlib.Path
    )->str:
    """
    Parse a c file and store into a json string
    """
    tree=c_file_to_json(filename)
    return json.dumps(tree)

def isUsingGcc()->bool:
    """
    Is the system using the GCC compiler
    """
    return False # TODO:

def isUsingCl()->bool:
    """
    Is the system using the windows CL compiler
    """
    return True # TODO:

def preprocessCFile(filename:pathlib.Path,
    includeDirs:typing.Iterable[pathlib.Path],
    defines:typing.Optional[typing.Dict[str,typing.Optional[str]]]=None,
    force:bool=False):
    """
    Run a c/cpp file through the preprocesor
    """
    preprocessedFilename=filename.with_suffix('.i')
    if force \
        or (not preprocessedFilename.exists()) \
        or filename.stat().st_mtime<preprocessedFilename.stat().st_mtime:
        #
        if isUsingGcc():
            cmd=['gcc','-E','-P']
            for inc in includeDirs:
                cmd.append(f'-I{inc}')
            cmd.extend([
                str(filename),'-o',str(preprocessedFilename)])
            print(cmd)
            raise NotImplementedError()
        elif isUsingCl():
            cmd=['cl','-nologo']
            if defines is not None:
                for k,v in defines.items():
                    if v is None:
                        cmd.append(f'-D{k}')
                    else:
                        cmd.append(f'-D{k}={v}')
            # extra defines to remove microsofty stuff from the generated .i 
            cmd.append(f'/FI{MSVC_STUBS_FILE}')
            cmd.extend(['/P','/EP'])
            for inc in includeDirs:
                cmd.append('/I')
                cmd.append(str(inc))
            cmd.extend([
                f'/Fi{preprocessedFilename}',
                str(filename)])
            print(cmd)
            po=subprocess.Popen(cmd,shell=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
            out,err=po.communicate()
            if err:
                print(err.decode('utf-8',errors='ignore'))
            elif out:
                print(out.decode('utf-8',errors='ignore'))
        else:
            raise FileNotFoundError('No compiler found')
    return preprocessedFilename

def enumCFiles(
    filesAndDirs=typing.Union[str,pathlib.Path,typing.Iterable[typing.Union[str,pathlib.Path]]]
    )->typing.Generator[pathlib.Path,None,None]:
    """
    Enumerate through all c/c++ files.

    You can specify files, glob expressions, or directories to recursively search
    """
    if isinstance(filesAndDirs,(str,pathlib.Path)):
        filesAndDirs=[filesAndDirs]
    for filename in filesAndDirs:
        if isinstance(filename,pathlib.Path):
            if filename.is_dir():
                yield from enumCFiles(filename.iterdir())
            elif filename.suffix in ('.c','.cpp','.cxx','.h','.hpp','.hxx'):
                yield filename
        elif filename.find('*')>=0:
            yield from enumCFiles(glob.glob(filename))
        else:
            yield from enumCFiles(pathlib.Path(filename))

def environmentIncludeDirs()->typing.Iterable[pathlib.Path]:
    """
    Get the include paths from the environment
    """
    environIncludes=os.environ.get('INCLUDE','').strip()
    if not environIncludes:
        return
    if os.name=='nt':
        includesList=environIncludes.split(';')
    else:
        includesList=environIncludes.split(':')
    for inc in includesList:
        yield pathlib.Path(inc)

def c_files_to_json_strings(
    filesAndDirs:typing.Union[str,pathlib.Path,typing.Iterable[typing.Union[str,pathlib.Path]]],
    includeDirs:typing.Optional[typing.Iterable[pathlib.Path]]=None,
    defines:typing.Optional[typing.Dict[str,typing.Optional[str]]]=None,
    useEnvironmentIncludes:bool=False
    )->typing.Dict[pathlib.Path,typing.Dict[str,typing.Any]]:
    """
    returns {filename:json}
    """
    ret:typing.Dict[pathlib.Path,typing.Dict[str,typing.Any]]={}
    source=[]
    includes=[]
    if includeDirs is None:
        includeDirs=[]
    includeDirs=set(includeDirs)
    if useEnvironmentIncludes:
        for inc in environmentIncludeDirs():
            includeDirs.add(inc)
    for filename in enumCFiles(filesAndDirs):
        if filename.suffix.startswith('.h'):
            includes.append(filename)
            includeDirs.add(filename.parent)
        else:
            source.append(filename)
    for filename in source:
        print(f'Parsing {filename}')
        ret[filename]=c_file_to_json(filename,includeDirs,defines)
    return ret

